In [42]:
import torch
from torch import nn

#creates the NN object. specifying the structure of the NN
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2,50), nn.Tanh(),
            nn.Linear(50,50),nn.Tanh(),
            nn.Linear(50,50),nn.Tanh(),
            nn.Linear(50,50),nn.Tanh(),
            nn.Linear(50,50),nn.Tanh(),
            nn.Linear(50,1),
        )
    def forward(self,x):
        u = self.net(x)
        return u
model = NeuralNetwork()

In [43]:
#set up the sampling amount for initial, boundary, residual
N_0, N_b, N_f = 100, 100, 5000


# initial position(x) is a 100 row, 1 column tensor object with float values from [0,1); time(t) for inital value is clearly 0
# Value(u) of the true initial condition is obtained by plugging the generated initial points into the given initial conditions 
# for initial condition

x_ic = torch.rand(N_0, 1)
t_ic = torch.zeros(N_0, 1)
u_true_ic = torch.sin(torch.pi * x_ic)


# boundary condition samples for time is scattered across [0,1), and for position it is half 0 and half 1, hence the tensor
# for x_bc is a concatenated [100,1] object

t_bc = torch.rand(N_b, 1)
x_bc = torch.cat([torch.zeros(N_b//2,1), torch.ones(N_b//2,1)])
# u_true_bc is zero, so no need to build it

#random sampling of the collocation points, requires_grad=True is a keyword argument that tells
#autograd to record operations involving that tensor, so derivatives with respect to it can be computed later.
x_f = torch.rand(N_f, 1, requires_grad=True)
t_f = torch.rand(N_f, 1, requires_grad=True)

#setting up the Adam optimizer. While the first parameter is the weights and biases of the NN "model" object 
#The second parameter is the hyperparameter: learning rate.
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

#choosing a loss function to optimize
loss_fn = nn.MSELoss()

#training loop, hyperparameter epoch is set to 10000
for epoch in range(10000):
    # for every new forward pass, sets the parameters' gradient to zero. So that gradients are not accumulated inter-epoch
    optimizer.zero_grad()
    #using model to predict the initial/boundary conditions
    #concatenate horizontally. Two [N,1] tensors become one [N,2]: column 0 is x, column 1 is t, matching nn.Linear(2, 50).
    #dim=0 would stack them vertically into [2N,1]
    u_pred_ic = model(torch.cat([x_ic, t_ic], dim=1))
    u_pred_bc = model(torch.cat([x_bc, t_bc], dim=1))
    # using the model to predict collocation values of u
    u = model(torch.cat([x_f, t_f], dim=1))
    #using automatic differentiation to calculate partial derivatives that are needed for residual calculation. Here,
    #the partial diff calculation for all 5000 u are handled at once. 

    u_t  = torch.autograd.grad(u, t_f, torch.ones_like(u), create_graph=True)[0]
    u_x  = torch.autograd.grad(u, x_f, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x_f, torch.ones_like(u_x), create_graph=True)[0]
    residual = u_t - u_xx

    #calculation of the mean squared loss of the initial condition, boundary conditon and PDE loss. 
    # "torch.zeros_like(residual)" a tensor of zeros, same shape as residual. It's the target: the residual should be zero 
    loss_ic  = loss_fn(u_pred_ic, u_true_ic)
    loss_bc  = loss_fn(u_pred_bc, torch.zeros_like(u_pred_bc))
    loss_pde = loss_fn(residual, torch.zeros_like(residual))

    loss = loss_ic + loss_bc + loss_pde
    
    #backprobagation and parameter update 
    loss.backward()
    optimizer.step()

    if epoch % 500 == 0:
        print(epoch, loss.item())

print(loss_ic.item(), loss_bc.item(), loss_pde.item())

0 0.46614357829093933
500 0.011229636147618294
1000 0.0008730439003556967
1500 0.0004526700358837843
2000 0.0003129428077954799
2500 0.00026147838798351586
3000 0.00017043855041265488
3500 0.00013418357411865145
4000 0.00010323870810680091
4500 8.289935067296028e-05
5000 6.530199607368559e-05
5500 6.190447311382741e-05
6000 4.7567551519023255e-05
6500 0.00027737891650758684
7000 4.199761679046787e-05
7500 5.64779074920807e-05
8000 0.0004130206652916968
8500 2.206489625677932e-05
9000 8.209886436816305e-05
9500 1.9282331777503714e-05
3.096420869042049e-06 7.107297278707847e-06 1.5240185348375235e-05


In [44]:
#Testing with analytical answer
#100 evenly spaced values from 0 to 1
x = torch.linspace(0, 1, 100)
t = torch.linspace(0, 1, 100)
X, T = torch.meshgrid(x, t, indexing='ij')
#reshapes the same 10,000 elements arranged as a single column, "-1" means "figure this out" 
x_eval = X.reshape(-1, 1)
t_eval = T.reshape(-1, 1)
"""the first two lines creates a shape 100 tensor with evenly spaced 100 values from 0 to 1,
the the third line creates a 100 by 100 tensor for X it is rows are the same, 
for T it is columns are the same. 
Then the two reshape just flattens both X and T into a 10000 row 1 column tensor"""

with torch.no_grad():
    #concatenates both [10000,1] tensors to get a valid input for the NN. the result is forward pass returned u value shape:[10000,1]
    u_pred  = model(torch.cat([x_eval, t_eval], dim=1))
    #calculation of the exact value using the analytical solution of the PDE
    u_exact = torch.sin(torch.pi * x_eval) * torch.exp(-torch.pi**2 * t_eval)
    #pairwise subtraction between pred and exact values then normalized, then relative error calculated
    error   = torch.norm(u_pred - u_exact) / torch.norm(u_exact)
print(error.item())

0.011688713915646076
